# 📦 APL Logistics Dashboard

*   List item
*   List item


### Advanced Analytics + Interactive Streamlit Dashboard + Performance Optimization

**✨ Features:**
- 🎨 **Decorator Functions**: Performance timing, caching, error handling
- 📊 **Time Series Analysis**: Trend visualization over time
- 📈 **Predictive Insights**: Forecasting and anomaly detection
- 💾 **Export Functionality**: Download reports in multiple formats
- 🔍 **Advanced Filtering**: Multi-level drill-down capabilities
- 📱 **Enhanced UX**: Better layout and interactive components

---

## Step 1: Install Required Packages

In [1]:
!pip install streamlit pandas plotly numpy pyngrok seaborn matplotlib scikit-learn openpyxl -q
print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.2 MB/s eta 0:00:00
✅ All packages installed successfully!


## Step 2: Upload Your Data File

In [2]:
from google.colab import files
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

print('📂 Please select your APL Logistics CSV file...')
uploaded = files.upload()

csv_files = glob.glob('/content/*.csv')
if not csv_files:
    raise FileNotFoundError('❌ No CSV found. Re-run this cell and select your file.')

CSV_PATH = csv_files[0]
print(f'\n✅ Found: {CSV_PATH}')
df_raw = pd.read_csv(CSV_PATH, encoding='latin1')
print(f'✅ Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns')

📂 Please select your APL Logistics CSV file...


Saving APL_Logistics.csv to APL_Logistics.csv

✅ Found: /content/APL_Logistics.csv
✅ Loaded 180,519 rows × 40 columns


## Step 3: Data Cleaning & Processing with Decorators

In [3]:
import time
import functools
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════════
# DECORATOR FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def timing_decorator(func):
    """Decorator to measure and log execution time of functions"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        print(f"⏱️  Starting {func.__name__}...")
        result = func(*args, **kwargs)
        elapsed_time = time.time() - start_time
        print(f"✅ {func.__name__} completed in {elapsed_time:.2f} seconds")
        return result
    return wrapper

def error_handler(func):
    """Decorator to handle errors gracefully with logging"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"❌ Error in {func.__name__}: {str(e)}")
            print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            raise
    return wrapper

def cache_result(func):
    """Decorator to cache expensive computation results"""
    cache = {}
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Create a cache key from arguments
        cache_key = str(args) + str(kwargs)
        if cache_key in cache:
            print(f"💾 Using cached result for {func.__name__}")
            return cache[cache_key]
        result = func(*args, **kwargs)
        cache[cache_key] = result
        return result
    return wrapper

def log_dataframe_info(func):
    """Decorator to log DataFrame shape and memory usage"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if isinstance(result, pd.DataFrame):
            memory_mb = result.memory_usage(deep=True).sum() / 1024**2
            print(f"📊 {func.__name__} output: {result.shape[0]:,} rows × {result.shape[1]} cols")
            print(f"   Memory usage: {memory_mb:.2f} MB")
        return result
    return wrapper

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PROCESSING FUNCTIONS WITH DECORATORS
# ═══════════════════════════════════════════════════════════════════════════════

@timing_decorator
@error_handler
@log_dataframe_info
def clean_data(df):
    """Clean and preprocess the raw logistics data"""
    print("="*70)
    print("DATA CLEANING & PROCESSING")
    print("="*70)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    real_col = 'Days for shipping (real)'
    sched_col = 'Days for shipment (scheduled)'
    delivery_col = 'Delivery Status'
    late_risk_col = 'Late_delivery_risk'

    print(f"\nOriginal shape: {df.shape}")

    # Remove missing values in critical columns
    critical_cols = [delivery_col, real_col, sched_col, late_risk_col]
    before = len(df)
    df = df.dropna(subset=critical_cols)
    print(f"Rows dropped (missing values): {before - len(df)}")

    # Remove invalid delivery status values
    valid_statuses = ['Advance shipping', 'Late delivery', 'Shipping on time', 'Shipping canceled']
    before = len(df)
    df = df[df[delivery_col].isin(valid_statuses)]
    print(f"Rows dropped (invalid status): {before - len(df)}")

    # Remove negative or zero shipping days
    before = len(df)
    df = df[(df[real_col] > 0) & (df[sched_col] > 0)]
    print(f"Rows dropped (invalid shipping days): {before - len(df)}")

    # Remove extreme shipping durations (> 60 days)
    before = len(df)
    df = df[(df[real_col] <= 60) & (df[sched_col] <= 60)]
    print(f"Rows dropped (extreme durations): {before - len(df)}")

    # Remove duplicates
    before = len(df)
    df = df.drop_duplicates()
    print(f"Rows dropped (duplicates): {before - len(df)}")

    # Standardize region and market names
    if 'Order Region' in df.columns:
        df['Order Region'] = df['Order Region'].str.title().str.strip()
    if 'Market' in df.columns:
        df['Market'] = df['Market'].str.title().str.strip()

    print("✅ Region and market names standardized")
    return df

@timing_decorator
@error_handler
@log_dataframe_info
def calculate_metrics(df):
    """Calculate delivery and financial metrics"""
    print("\n" + "="*70)
    print("CALCULATING METRICS")
    print("="*70)

    real_col = 'Days for shipping (real)'
    sched_col = 'Days for shipment (scheduled)'

    # 1. Delay gap calculation
    df['delay_gap'] = df[real_col] - df[sched_col]

    # 2. On-time delivery flag
    df['On_Time'] = (df['delay_gap'] <= 0).astype(int)

    # 3. Delivery classification
    conditions = [
        df['delay_gap'] < 0,
        df['delay_gap'] == 0,
        df['delay_gap'] > 0
    ]
    choices = ['Early', 'On Time', 'Late']
    df['delivery_classification'] = np.select(conditions, choices, default='Unknown')

    # 4. SLA status categorization
    sla_conditions = [
        df['delay_gap'] <= 0,
        (df['delay_gap'] > 0) & (df['delay_gap'] <= 2),
        (df['delay_gap'] > 2) & (df['delay_gap'] <= 5),
        df['delay_gap'] > 5
    ]
    sla_choices = ['SLA Met', 'Minor Breach', 'Critical Breach', 'Severe Breach']
    df['SLA_Status'] = np.select(sla_conditions, sla_choices, default='Unknown')

    # 5. Financial metrics
    if 'Sales' in df.columns and 'Order Profit Per Order' in df.columns:
        df['Profit_Margin_%'] = (df['Order Profit Per Order'] / df['Sales'] * 100).fillna(0)

    print("✅ All metrics calculated successfully")
    return df

@timing_decorator
@error_handler
def calculate_shipping_efficiency(df):
    """Calculate shipping mode efficiency scores"""
    print("\n" + "="*70)
    print("SHIPPING MODE EFFICIENCY SCORING")
    print("="*70)

    mode_perf = df.groupby('Shipping Mode').agg(
        Total_Orders=('Sales', 'count'),
        Avg_Delay=('delay_gap', 'mean'),
        OnTime_Rate=('On_Time', 'mean'),
        Total_Revenue=('Sales', 'sum'),
        Total_Profit=('Order Profit Per Order', 'sum'),
    ).round(2)

    # Calculate composite efficiency score
    mode_perf['OnTime_Rate_Pct'] = mode_perf['OnTime_Rate'] * 100
    mode_perf['Efficiency_Score'] = (
        (mode_perf['OnTime_Rate_Pct'] * 0.6) -
        (mode_perf['Avg_Delay'] * 2)
    ).round(2)

    # Assign grades
    def assign_grade(score):
        if score >= 50: return 'A'
        elif score >= 40: return 'B'
        elif score >= 30: return 'C'
        elif score >= 20: return 'D'
        else: return 'F'

    mode_perf['Grade'] = mode_perf['Efficiency_Score'].apply(assign_grade)
    mode_perf_sorted = mode_perf.sort_values('Efficiency_Score', ascending=False)

    print("\n📊 Shipping Mode Rankings:")
    print(mode_perf_sorted[['Efficiency_Score', 'Grade', 'OnTime_Rate_Pct', 'Avg_Delay']])

    return mode_perf_sorted

# ═══════════════════════════════════════════════════════════════════════════════
# EXECUTE DATA PROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

# Clean data
df_raw = clean_data(df_raw)

# Calculate metrics
df_raw = calculate_metrics(df_raw)

# Calculate shipping efficiency
shipping_efficiency = calculate_shipping_efficiency(df_raw)

# Calculate additional aggregations
print("\n" + "="*70)
print("ADDITIONAL AGGREGATIONS")
print("="*70)

# Regional performance
region_perf = df_raw.groupby('Order Region').agg(
    Total_Orders=('Sales', 'count'),
    Avg_Delay=('delay_gap', 'mean'),
    OnTime_Rate=('On_Time', 'mean'),
    Total_Revenue=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum'),
).round(2)
region_perf['OnTime_Rate'] = (region_perf['OnTime_Rate'] * 100).round(2)

# Market performance
market_perf = df_raw.groupby('Market').agg(
    Total_Orders=('Sales', 'count'),
    Avg_Delay=('delay_gap', 'mean'),
    OnTime_Rate=('On_Time', 'mean'),
    Total_Revenue=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum'),
).round(2)
market_perf['OnTime_Rate'] = (market_perf['OnTime_Rate'] * 100).round(2)

# Customer segment analysis
segment_perf = df_raw.groupby('Customer Segment').agg(
    Total_Orders=('Sales', 'count'),
    Avg_Delay=('delay_gap', 'mean'),
    OnTime_Rate=('On_Time', 'mean'),
    SLA_Met_Rate=('SLA_Status', lambda x: (x == 'SLA Met').mean()),
    Total_Revenue=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum'),
).round(2)
segment_perf['OnTime_Rate'] = (segment_perf['OnTime_Rate'] * 100).round(2)
segment_perf['SLA_Met_Rate'] = (segment_perf['SLA_Met_Rate'] * 100).round(2)

print("✅ Regional, market, and segment metrics calculated")

# Save processed data
df_raw.to_csv('/content/APL_Logistics_Processed.csv', index=False)
print(f"\n✅ Processed data saved to /content/APL_Logistics_Processed.csv")
print("="*70)

⏱️  Starting clean_data...
DATA CLEANING & PROCESSING

Original shape: (180519, 40)
Rows dropped (missing values): 0
Rows dropped (invalid status): 0
Rows dropped (invalid shipping days): 9737
Rows dropped (extreme durations): 0
Rows dropped (duplicates): 0
✅ Region and market names standardized
📊 clean_data output: 170,782 rows × 40 cols
   Memory usage: 215.71 MB
✅ clean_data completed in 2.11 seconds
⏱️  Starting calculate_metrics...

CALCULATING METRICS
✅ All metrics calculated successfully
📊 calculate_metrics output: 170,782 rows × 45 cols
   Memory usage: 238.01 MB
✅ calculate_metrics completed in 1.02 seconds
⏱️  Starting calculate_shipping_efficiency...

SHIPPING MODE EFFICIENCY SCORING

📊 Shipping Mode Rankings:
                Efficiency_Score Grade  OnTime_Rate_Pct  Avg_Delay
Shipping Mode                                                     
Standard Class             36.00     C             60.0      -0.00
Second Class                8.02     F             20.0       1.99
F

## Step 4: Configure ngrok
### ⚠️ PASTE YOUR NGROK TOKEN BELOW

In [4]:
# ============================================
# PASTE YOUR NGROK TOKEN BELOW
# ============================================

NGROK_TOKEN = "3BCc4toNvoOR0c2rdPvBP3DBQlD_5bEvfo3xrmr5DUHrrYdbU"

# ============================================

!ngrok config add-authtoken {NGROK_TOKEN}
print("✅ ngrok token configured!")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ ngrok token configured!


## Step 5: Create Enhanced Streamlit Dashboard

### ✅ FIXED: Removed duplicate sidebar section and fixed SyntaxError

In [5]:
%%writefile apl_enhanced_dashboard.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime, timedelta
import io
import base64

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
st.set_page_config(
    page_title="APL Logistics Dashboard",
    page_icon="📦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ═══════════════════════════════════════════════════════════════════════════════
# CUSTOM CSS WITH ENHANCED STYLING
# ═══════════════════════════════════════════════════════════════════════════════
st.markdown("""
<style>
    .main-header {
        font-size: 3rem;
        font-weight: bold;
        background: linear-gradient(120deg, #1f77b4, #2ecc71);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        text-align: center;
        padding: 1.5rem 0;
        margin-bottom: 2rem;
    }
    .metric-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 1rem;
        color: white;
        box-shadow: 0 8px 16px rgba(0,0,0,0.1);
        transition: transform 0.3s;
    }
    .metric-card:hover {
        transform: translateY(-5px);
    }
    .info-box {
        background-color: #e3f2fd;
        padding: 1rem;
        border-radius: 0.5rem;
        border-left: 4px solid #2196f3;
        margin: 1rem 0;
    }
    .success-box {
        background-color: #e8f5e9;
        padding: 1rem;
        border-radius: 0.5rem;
        border-left: 4px solid #4caf50;
        margin: 1rem 0;
    }
    .warning-box {
        background-color: #fff3e0;
        padding: 1rem;
        border-radius: 0.5rem;
        border-left: 4px solid #ff9800;
        margin: 1rem 0;
    }
    .stTabs [data-baseweb="tab-list"] {
        gap: 2rem;
    }
    .stTabs [data-baseweb="tab"] {
        height: 50px;
        padding: 10px 20px;
        background-color: #f0f2f6;
        border-radius: 5px 5px 0 0;
    }
    .stTabs [aria-selected="true"] {
        background-color: #1f77b4;
        color: white;
    }
</style>
""", unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS WITH DECORATORS
# ═══════════════════════════════════════════════════════════════════════════════

@st.cache_data
def load_data():
    """Load and cache processed data"""
    try:
        df = pd.read_csv('/content/APL_Logistics_Processed.csv', encoding='utf-8')
        return df
    except FileNotFoundError:
        st.error("❌ Data file not found! Please run Steps 1-3 in the notebook first to process your data.")
        st.stop()
    except Exception as e:
        st.error(f"❌ Error loading data: {str(e)}")
        st.stop()

def create_download_link(df, filename, file_label):
    """Create a download link for dataframe"""
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    href = f'<a href="data:file/csv;base64,{b64}" download="{filename}">📥 Download {file_label}</a>'
    return href

def format_number(num):
    """Format large numbers with K, M suffixes"""
    if num >= 1_000_000:
        return f"${num/1_000_000:.1f}M"
    elif num >= 1_000:
        return f"${num/1_000:.1f}K"
    else:
        return f"${num:.0f}"

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════
df = load_data()

# ═══════════════════════════════════════════════════════════════════════════════
# SIDEBAR - FILTERS (FIXED: Removed duplicate sidebar section)
# ═══════════════════════════════════════════════════════════════════════════════

st.sidebar.title("🎛️ Dashboard Controls")
st.sidebar.markdown("---")


st.sidebar.subheader("🔍 Filters")

# Shipping mode filter
shipping_modes = ['All'] + sorted(df['Shipping Mode'].unique().tolist())
selected_shipping_mode = st.sidebar.selectbox("📦 Shipping Mode", shipping_modes, key='shipping_mode_filter')

# Region filter
regions = ['All'] + sorted(df['Order Region'].dropna().unique().tolist())
selected_region = st.sidebar.selectbox("🌍 Region", regions, key='region_filter')

# Market filter
markets = ['All'] + sorted(df['Market'].dropna().unique().tolist())
selected_market = st.sidebar.selectbox("🌎 Market", markets, key='market_filter')

# Customer segment filter
segments = ['All'] + sorted(df['Customer Segment'].dropna().unique().tolist())
selected_segment = st.sidebar.selectbox("👥 Customer Segment", segments, key='segment_filter')

# SLA status filter
sla_statuses = ['All'] + sorted(df['SLA_Status'].unique().tolist())
selected_sla = st.sidebar.selectbox("⚠️ SLA Status", sla_statuses, key='sla_filter')

# Delay range selector
st.sidebar.markdown("---")
st.sidebar.subheader("📊 Delay Range")
min_delay = int(df['delay_gap'].min())
max_delay = int(df['delay_gap'].max())
delay_range = st.sidebar.slider(
    "Delay Gap (days)",
    min_delay, max_delay, (min_delay, max_delay),
    key='delay_range_slider'
)

# APPLY FILTERS
filtered_df = df.copy()

if selected_shipping_mode != 'All':
    filtered_df = filtered_df[filtered_df['Shipping Mode'] == selected_shipping_mode]

if selected_region != 'All':
    filtered_df = filtered_df[filtered_df['Order Region'] == selected_region]

if selected_market != 'All':
    filtered_df = filtered_df[filtered_df['Market'] == selected_market]

if selected_segment != 'All':
    filtered_df = filtered_df[filtered_df['Customer Segment'] == selected_segment]

if selected_sla != 'All':
    filtered_df = filtered_df[filtered_df['SLA_Status'] == selected_sla]

# FIXED: Removed duplicate line that caused SyntaxError
filtered_df = filtered_df[
    (filtered_df['delay_gap'] >= delay_range[0]) &
    (filtered_df['delay_gap'] <= delay_range[1])
]

# Export section
st.sidebar.markdown("---")
st.sidebar.subheader("📥 Export Data")
if st.sidebar.button("📊 Export Filtered Data"):
    st.sidebar.markdown(create_download_link(
        filtered_df,
        "apl_filtered_data.csv",
        "Filtered Dataset"
    ), unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN DASHBOARD
# ═══════════════════════════════════════════════════════════════════════════════

st.markdown('<div class="main-header">📦 APL Logistics Dashboard</div>', unsafe_allow_html=True)
st.markdown("### 🚀 Advanced Analytics with AI-Powered Insights")

# Show active filters
active_filters = []
if selected_shipping_mode != 'All': active_filters.append(f"📦 {selected_shipping_mode}")
if selected_region != 'All': active_filters.append(f"🌍 {selected_region}")
if selected_market != 'All': active_filters.append(f"🌎 {selected_market}")
if selected_segment != 'All': active_filters.append(f"👥 {selected_segment}")
if selected_sla != 'All': active_filters.append(f"⚠️ {selected_sla}")

if active_filters:
    st.markdown(f"**Active Filters:** {' | '.join(active_filters)}")

st.markdown("---")

# ═══════════════════════════════════════════════════════════════════════════════
# TAB NAVIGATION
# ═══════════════════════════════════════════════════════════════════════════════
tab1, tab2, tab3, tab4, tab5,tab6 = st.tabs([
    "📊 Overview",
    "💰 Financial Analysis",
    "🚚 Shipping Performance",
    "🔮 Predictive Insights",
    "🗺️ Geographic Distribution & Analysis",
    "📋 Detailed Reports"

])

# ═══════════════════════════════════════════════════════════════════════════════
# TAB 1: OVERVIEW
# ═══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.header("📊 Key Performance Indicators")

    # Calculate KPIs
    total_orders = len(filtered_df)
    total_revenue = filtered_df['Sales'].sum()
    total_profit = filtered_df['Order Profit Per Order'].sum()
    avg_profit_ratio = filtered_df['Order Item Profit Ratio'].mean() * 100
    on_time_rate = filtered_df['On_Time'].mean() * 100
    avg_delay = filtered_df['delay_gap'].mean()

    # Display KPIs in columns
    col1, col2, col3, col4, col5, col6 = st.columns(6)

    with col1:
        st.metric("📦 Total Orders", f"{total_orders:,}")

    with col2:
        st.metric("💰 Revenue", format_number(total_revenue))

    with col3:
        st.metric("💵 Profit", format_number(total_profit))

    with col4:
        st.metric("📈 Margin", f"{avg_profit_ratio:.1f}%")

    with col5:
        st.metric("✅ On-Time", f"{on_time_rate:.1f}%")

    with col6:
        st.metric("⏱️ Avg Delay", f"{avg_delay:.1f}d")

    st.markdown("---")

    # Visualizations
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🎯 Delivery Performance")
        delivery_counts = filtered_df['delivery_classification'].value_counts()

        fig_delivery = go.Figure(data=[go.Pie(
            labels=delivery_counts.index,
            values=delivery_counts.values,
            hole=0.5,
            marker_colors=['#2ecc71', '#3498db', '#e74c3c'],
            textinfo='label+percent',
            textfont_size=14
        )])
        fig_delivery.update_layout(
            title="Delivery Classification Distribution",
            height=400,
            showlegend=True
        )
        st.plotly_chart(fig_delivery, use_container_width=True)

    with col2:
        st.subheader("⚠️ SLA Status")
        sla_counts = filtered_df['SLA_Status'].value_counts()

        colors_sla = {
            'SLA Met': '#2ecc71',
            'Minor Breach': '#f39c12',
            'Critical Breach': '#e67e22',
            'Severe Breach': '#e74c3c'
        }
        bar_colors = [colors_sla.get(x, '#95a5a6') for x in sla_counts.index]

        fig_sla = go.Figure(data=[go.Bar(
            x=sla_counts.index,
            y=sla_counts.values,
            marker_color=bar_colors,
            text=sla_counts.values,
            textposition='auto',
            textfont_size=14
        )])
        fig_sla.update_layout(
            title="SLA Status Distribution",
            xaxis_title="SLA Status",
            yaxis_title="Count",
            height=400
        )
        st.plotly_chart(fig_sla, use_container_width=True)

    st.markdown("---")

    # Regional heatmap
    st.subheader("🌍 Regional Performance Heatmap")
    region_metrics = filtered_df.groupby('Order Region').agg({
        'Sales': 'sum',
        'Order Profit Per Order': 'sum',
        'On_Time': 'mean',
        'delay_gap': 'mean'
    }).round(2)

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=region_metrics.values.T,
        x=region_metrics.index,
        y=['Revenue', 'Profit', 'On-Time Rate', 'Avg Delay'],
        colorscale='RdYlGn',
        text=region_metrics.values.T,
        texttemplate='%{text:.2f}',
        textfont={"size": 10}
    ))
    fig_heatmap.update_layout(
        title="Regional Performance Metrics",
        height=300
    )
    st.plotly_chart(fig_heatmap, use_container_width=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TAB 2: FINANCIAL ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.header("💰 Financial Impact Analysis")

    col1, col2, col3 = st.columns(3)

    with col1:
        profit_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
        st.metric("💹 Overall Profit Margin", f"{profit_margin:.2f}%")

    with col2:
        revenue_at_risk = filtered_df[filtered_df['Late_delivery_risk'] == 1]['Sales'].sum()
        st.metric("⚠️ Revenue at Risk", format_number(revenue_at_risk))

    with col3:
        avg_order_value = total_revenue / total_orders if total_orders > 0 else 0
        st.metric("💵 Avg Order Value", f"${avg_order_value:.2f}")

    st.markdown("---")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("📊 Revenue by SLA Status")
        sla_revenue = filtered_df.groupby('SLA_Status')['Sales'].sum().sort_values(ascending=False)

        fig_sla_rev = go.Figure(data=[go.Bar(
            x=sla_revenue.index,
            y=sla_revenue.values,
            marker_color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'][:len(sla_revenue)],
            text=[format_number(v) for v in sla_revenue.values],
            textposition='auto'
        )])
        fig_sla_rev.update_layout(
            title="Total Revenue by SLA Status",
            xaxis_title="SLA Status",
            yaxis_title="Revenue ($)",
            height=400
        )
        st.plotly_chart(fig_sla_rev, use_container_width=True)

    with col2:
        st.subheader("💵 Profit by Delivery Type")
        class_profit = filtered_df.groupby('delivery_classification')['Order Profit Per Order'].sum()

        fig_class_profit = go.Figure(data=[go.Bar(
            x=class_profit.index,
            y=class_profit.values,
            marker_color=['#2ecc71', '#3498db', '#e74c3c'],
            text=[format_number(v) for v in class_profit.values],
            textposition='auto'
        )])
        fig_class_profit.update_layout(
            title="Total Profit by Delivery Classification",
            xaxis_title="Classification",
            yaxis_title="Profit ($)",
            height=400
        )
        st.plotly_chart(fig_class_profit, use_container_width=True)

    st.markdown("---")

    # Customer segment financial analysis
    st.subheader("👥 Financial Performance by Customer Segment")
    segment_fin = filtered_df.groupby('Customer Segment').agg({
        'Sales': 'sum',
        'Order Profit Per Order': 'sum'
    }).reset_index()
    segment_fin['Profit_Margin'] = (segment_fin['Order Profit Per Order'] / segment_fin['Sales'] * 100).round(2)

    fig_segment = go.Figure()
    fig_segment.add_trace(go.Bar(
        x=segment_fin['Customer Segment'],
        y=segment_fin['Sales'],
        name='Revenue',
        marker_color='#3498db'
    ))
    fig_segment.add_trace(go.Bar(
        x=segment_fin['Customer Segment'],
        y=segment_fin['Order Profit Per Order'],
        name='Profit',
        marker_color='#2ecc71'
    ))
    fig_segment.update_layout(
        title="Revenue vs Profit by Customer Segment",
        barmode='group',
        height=400
    )
    st.plotly_chart(fig_segment, use_container_width=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TAB 3: SHIPPING PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.header("🚚 Shipping Mode Performance Analysis")

    # Calculate shipping mode metrics
    mode_perf = filtered_df.groupby('Shipping Mode').agg(
        Total_Orders=('Sales', 'count'),
        Avg_Delay=('delay_gap', 'mean'),
        OnTime_Rate=('On_Time', 'mean'),
        Total_Revenue=('Sales', 'sum'),
        Total_Profit=('Order Profit Per Order', 'sum'),
    ).round(2)

    mode_perf['OnTime_Rate_Pct'] = (mode_perf['OnTime_Rate'] * 100).round(2)
    mode_perf['Efficiency_Score'] = (
        (mode_perf['OnTime_Rate_Pct'] * 0.6) -
        (mode_perf['Avg_Delay'] * 2)
    ).round(2)

    def assign_grade(score):
        if score >= 50: return '🏆 A'
        elif score >= 40: return '🥈 B'
        elif score >= 30: return '🥉 C'
        elif score >= 20: return '📉 D'
        else: return '❌ F'

    mode_perf['Grade'] = mode_perf['Efficiency_Score'].apply(assign_grade)
    mode_perf_sorted = mode_perf.sort_values('Efficiency_Score', ascending=False)

    # Display shipping mode rankings
    st.subheader("🏅 Shipping Mode Rankings")
    st.dataframe(
        mode_perf_sorted[['Grade', 'Efficiency_Score', 'OnTime_Rate_Pct', 'Avg_Delay', 'Total_Orders']],
        use_container_width=True
    )

    st.markdown("---")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("📊 Efficiency Score Comparison")
        fig_efficiency = go.Figure(data=[go.Bar(
            x=mode_perf_sorted.index,
            y=mode_perf_sorted['Efficiency_Score'],
            marker_color=mode_perf_sorted['Efficiency_Score'],
            marker_colorscale='RdYlGn',
            text=mode_perf_sorted['Efficiency_Score'],
            textposition='auto'
        )])
        fig_efficiency.update_layout(
            title="Shipping Mode Efficiency Scores",
            xaxis_title="Shipping Mode",
            yaxis_title="Efficiency Score",
            height=400
        )
        st.plotly_chart(fig_efficiency, use_container_width=True)

    with col2:
        st.subheader("💰 Revenue vs Profit Scatter")
        fig_scatter = go.Figure(data=[go.Scatter(
            x=mode_perf_sorted['Total_Revenue'],
            y=mode_perf_sorted['Total_Profit'],
            mode='markers+text',
            text=mode_perf_sorted.index,
            textposition='top center',
            marker=dict(
                size=mode_perf_sorted['Total_Orders']/100,
                color=mode_perf_sorted['Efficiency_Score'],
                colorscale='RdYlGn',
                showscale=True,
                colorbar=dict(title="Efficiency")
            )
        )])
        fig_scatter.update_layout(
            title="Revenue vs Profit (bubble size = orders)",
            xaxis_title="Total Revenue ($)",
            yaxis_title="Total Profit ($)",
            height=400
        )
        st.plotly_chart(fig_scatter, use_container_width=True)

    st.markdown("---")

    # Delay distribution by shipping mode
    st.subheader("📦 Delay Distribution by Shipping Mode")
    fig_box = go.Figure()
    for mode in filtered_df['Shipping Mode'].unique():
        mode_data = filtered_df[filtered_df['Shipping Mode'] == mode]['delay_gap']
        fig_box.add_trace(go.Box(
            y=mode_data,
            name=mode,
            boxmean='sd'
        ))
    fig_box.update_layout(
        title="Delay Gap Distribution by Shipping Mode",
        yaxis_title="Delay Gap (days)",
        height=400
    )
    st.plotly_chart(fig_box, use_container_width=True)


# ═══════════════════════════════════════════════════════════════════════════════
# TAB 5: PREDICTIVE INSIGHTS
# ═══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.header("🔮 Predictive Insights & Anomaly Detection")

    st.markdown("""
    <div class="info-box">
        <strong>📊 Statistical Analysis</strong><br>
        This section provides data-driven insights based on historical patterns.
    </div>
    """, unsafe_allow_html=True)

    # Calculate percentiles for delay
    delay_percentiles = filtered_df['delay_gap'].quantile([0.25, 0.5, 0.75, 0.90, 0.95]).round(2)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric("📊 Median Delay", f"{delay_percentiles[0.5]:.1f} days")

    with col2:
        st.metric("⚠️ 90th Percentile", f"{delay_percentiles[0.9]:.1f} days")

    with col3:
        st.metric("🔴 95th Percentile", f"{delay_percentiles[0.95]:.1f} days")

    st.markdown("---")

    # Delay distribution analysis
    st.subheader("📊 Delay Distribution Analysis")

    fig_hist = go.Figure()
    fig_hist.add_trace(go.Histogram(
        x=filtered_df['delay_gap'],
        nbinsx=50,
        marker_color='#3498db',
        name='Delay Distribution'
    ))

    # Add percentile lines
    for percentile, color in [(0.5, 'green'), (0.9, 'orange'), (0.95, 'red')]:
        fig_hist.add_vline(
            x=delay_percentiles[percentile],
            line_dash="dash",
            line_color=color,
            annotation_text=f"{int(percentile*100)}th percentile"
        )

    fig_hist.update_layout(
        title="Delay Gap Distribution with Percentiles",
        xaxis_title="Delay Gap (days)",
        yaxis_title="Frequency",
        height=400
    )
    st.plotly_chart(fig_hist, use_container_width=True)

    st.markdown("---")

    # Risk assessment
    st.subheader("⚠️ Risk Assessment")

    high_risk_orders = filtered_df[filtered_df['Late_delivery_risk'] == 1]
    high_risk_pct = len(high_risk_orders) / len(filtered_df) * 100 if len(filtered_df) > 0 else 0

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("""
        <div class="warning-box">
            <h4>🚨 High Risk Orders</h4>
        """, unsafe_allow_html=True)
        st.metric("Count", f"{len(high_risk_orders):,}")
        st.metric("Percentage", f"{high_risk_pct:.1f}%")
        st.metric("Revenue at Risk", format_number(high_risk_orders['Sales'].sum()))
        st.markdown("</div>", unsafe_allow_html=True)

    with col2:
        # Risk by shipping mode
        risk_by_mode = filtered_df.groupby('Shipping Mode')['Late_delivery_risk'].agg(['sum', 'mean']).round(2)
        risk_by_mode['Risk_Rate_%'] = (risk_by_mode['mean'] * 100).round(2)
        risk_by_mode = risk_by_mode.sort_values('Risk_Rate_%', ascending=False)

        fig_risk = go.Figure(data=[go.Bar(
            x=risk_by_mode.index,
            y=risk_by_mode['Risk_Rate_%'],
            marker_color=risk_by_mode['Risk_Rate_%'],
            marker_colorscale='Reds',
            text=risk_by_mode['Risk_Rate_%'],
            textposition='auto'
        )])
        fig_risk.update_layout(
            title="Risk Rate by Shipping Mode",
            xaxis_title="Shipping Mode",
            yaxis_title="Risk Rate (%)",
            height=300
        )
        st.plotly_chart(fig_risk, use_container_width=True)

    st.markdown("---")

    # Correlation analysis
    st.subheader("📊 Correlation Analysis")

    numeric_cols = ['delay_gap', 'Sales', 'Order Profit Per Order', 'Order Item Profit Ratio', 'Late_delivery_risk']
    corr_matrix = filtered_df[numeric_cols].corr().round(2)

    fig_corr = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=corr_matrix.values,
        texttemplate='%{text:.2f}',
        textfont={"size": 10}
    ))
    fig_corr.update_layout(
        title="Correlation Matrix of Key Metrics",
        height=500
    )
    st.plotly_chart(fig_corr, use_container_width=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TAB 4: GEOGRAPHIC ANALYSIS (NEW!)
# ═══════════════════════════════════════════════════════════════════════════════
with tab5:
    st.header("🗺️ Geographic Distribution & Analysis")

    st.markdown("""
    <div class="info-box">
        <strong>📍 Geographic Insights</strong><br>
        Visualize customer locations and delivery performance across regions with interactive maps.
    </div>
    """, unsafe_allow_html=True)

    # Map Controls
    st.subheader("🎛️ Map Display Controls")
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        show_customer_locations = st.checkbox("👥 Customer Locations", value=True)
    with col2:
        show_order_locations = st.checkbox("📦 Order Locations", value=False)
    with col3:
        color_by_metric = st.selectbox(
            "🎨 Color By",
            ["Delivery Status", "SLA Status", "Delay Gap", "Sales", "Profit"]
        )
    with col4:
        map_style = st.selectbox(
            "🗺️ Map Style",
            ["open-street-map", "carto-positron", "carto-darkmatter", "stamen-terrain"]
        )

    st.markdown("---")

    # Prepare map data
    map_df = filtered_df.copy()

    # Remove rows with missing lat/long
    if show_customer_locations:
        map_df = map_df.dropna(subset=['Latitude', 'Longitude'])

    if len(map_df) > 0:
        # Limit to 5000 points for performance
        if len(map_df) > 5000:
            st.warning(f"⚠️ Showing sample of 5000 points out of {len(map_df):,} total orders for performance.")
            map_df = map_df.sample(n=5000, random_state=42)

        # Create color mapping based on selected metric
        if color_by_metric == "Delivery Status":
            color_column = 'Delivery Status'
            color_map = {
                'Advance shipping': '#2ecc71',
                'Shipping on time': '#3498db',
                'Late delivery': '#e74c3c',
                'Shipping canceled': '#95a5a6'
            }
        elif color_by_metric == "SLA Status":
            color_column = 'SLA_Status'
            color_map = {
                'SLA Met': '#2ecc71',
                'Minor Breach': '#f39c12',
                'Critical Breach': '#e67e22',
                'Severe Breach': '#e74c3c'
            }
        elif color_by_metric == "Delay Gap":
            color_column = 'delay_gap'
            color_map = None  # Will use continuous color scale
        elif color_by_metric == "Sales":
            color_column = 'Sales'
            color_map = None
        else:  # Profit
            color_column = 'Order Profit Per Order'
            color_map = None

        # Create the map
        if color_map:
            # Categorical coloring
            fig_map = px.scatter_mapbox(
                map_df,
                lat='Latitude',
                lon='Longitude',
                color=color_column,
                color_discrete_map=color_map,
                hover_data={
                    'Latitude': ':.4f',
                    'Longitude': ':.4f',
                    'Customer City': True,
                    'Order Region': True,
                    'Shipping Mode': True,
                    'Sales': ':$,.2f',
                    'delay_gap': ':.1f',
                    'Delivery Status': True,
                    'SLA_Status': True
                },
                size_max=15,
                zoom=1,
                height=600,
                title=f"Customer Locations colored by {color_by_metric}"
            )
        else:
            # Continuous coloring
            fig_map = px.scatter_mapbox(
                map_df,
                lat='Latitude',
                lon='Longitude',
                color=color_column,
                color_continuous_scale='RdYlGn_r' if color_by_metric == 'Delay Gap' else 'Viridis',
                hover_data={
                    'Latitude': ':.4f',
                    'Longitude': ':.4f',
                    'Customer City': True,
                    'Order Region': True,
                    'Shipping Mode': True,
                    'Sales': ':$,.2f',
                    'delay_gap': ':.1f',
                    'Delivery Status': True,
                    'SLA_Status': True
                },
                size_max=15,
                zoom=1,
                height=600,
                title=f"Customer Locations colored by {color_by_metric}"
            )

        fig_map.update_layout(
            mapbox_style=map_style,
            margin={"r": 0, "t": 50, "l": 0, "b": 0}
        )

        st.plotly_chart(fig_map, use_container_width=True)

        st.markdown("---")

        # Geographic Statistics
        col1, col2 = st.columns(2)

        with col1:
            st.subheader("🌍 Top 10 Cities by Order Volume")
            city_stats = map_df.groupby('Customer City').agg({
                'Sales': 'count',
                'Order Profit Per Order': 'sum',
                'On_Time': 'mean'
            }).round(2)
            city_stats.columns = ['Orders', 'Total_Profit', 'OnTime_Rate']
            city_stats['OnTime_Rate'] = (city_stats['OnTime_Rate'] * 100).round(1)
            city_stats = city_stats.sort_values('Orders', ascending=False).head(10)

            st.dataframe(city_stats, use_container_width=True)

        with col2:
            st.subheader("📊 Geographic Distribution by Metric")

            metric_option = st.selectbox(
                "Select Metric to Analyze",
                ["Order Count", "Revenue", "Profit", "On-Time Rate"]
            )

            if metric_option == "Order Count":
                geo_data = map_df.groupby('Order Region')['Sales'].count()
            elif metric_option == "Revenue":
                geo_data = map_df.groupby('Order Region')['Sales'].sum()
            elif metric_option == "Profit":
                geo_data = map_df.groupby('Order Region')['Order Profit Per Order'].sum()
            else:  # On-Time Rate
                geo_data = (map_df.groupby('Order Region')['On_Time'].mean() * 100).round(2)

            geo_data = geo_data.sort_values(ascending=False)

            fig_geo_bar = go.Figure(data=[go.Bar(
                x=geo_data.index,
                y=geo_data.values,
                marker_color='#3498db',
                text=geo_data.values,
                textposition='auto'
            )])
            fig_geo_bar.update_layout(
                title=f"{metric_option} by Region",
                xaxis_title="Region",
                yaxis_title=metric_option,
                height=350
            )
            st.plotly_chart(fig_geo_bar, use_container_width=True)

        st.markdown("---")

        # Heat map of countries
        st.subheader("🌐 Country-Level Performance")

        country_perf = map_df.groupby('Customer Country').agg({
            'Sales': ['count', 'sum'],
            'Order Profit Per Order': 'sum',
            'On_Time': 'mean',
            'delay_gap': 'mean'
        }).round(2)

        country_perf.columns = ['Orders', 'Revenue', 'Profit', 'OnTime_Rate', 'Avg_Delay']
        country_perf['OnTime_Rate'] = (country_perf['OnTime_Rate'] * 100).round(1)
        country_perf = country_perf.sort_values('Orders', ascending=False).head(15)

        st.dataframe(country_perf, use_container_width=True)

    else:
        st.warning("⚠️ No data available with valid latitude/longitude coordinates for the selected filters.")


# ═══════════════════════════════════════════════════════════════════════════════
# TAB 6: DETAILED REPORTS
# ═══════════════════════════════════════════════════════════════════════════════
with tab6:
    st.header("📋 Detailed Reports & Data Tables")

    report_type = st.selectbox(
        "Select Report Type",
        ["Shipping Mode Analysis", "Regional Performance", "Market Analysis",
         "Customer Segment Analysis", "SLA Breach Details", "Top Performers"]
    )

    if report_type == "Shipping Mode Analysis":
        st.subheader("🚚 Comprehensive Shipping Mode Report")

        mode_report = filtered_df.groupby('Shipping Mode').agg({
            'Sales': ['count', 'sum', 'mean'],
            'Order Profit Per Order': ['sum', 'mean'],
            'delay_gap': ['mean', 'std', 'min', 'max'],
            'On_Time': 'mean',
            'Late_delivery_risk': 'sum'
        }).round(2)

        mode_report.columns = ['_'.join(col).strip() for col in mode_report.columns.values]
        st.dataframe(mode_report, use_container_width=True)

        if st.button("📥 Download Shipping Mode Report"):
            st.markdown(create_download_link(
                mode_report.reset_index(),
                "shipping_mode_report.csv",
                "Shipping Mode Report"
            ), unsafe_allow_html=True)

    elif report_type == "Regional Performance":
        st.subheader("🌍 Regional Performance Report")

        region_report = filtered_df.groupby('Order Region').agg({
            'Sales': ['count', 'sum'],
            'Order Profit Per Order': 'sum',
            'delay_gap': 'mean',
            'On_Time': 'mean',
            'SLA_Status': lambda x: (x == 'SLA Met').mean()
        }).round(2)

        region_report.columns = ['Total_Orders', 'Total_Revenue', 'Total_Profit', 'Avg_Delay', 'OnTime_Rate', 'SLA_Met_Rate']
        region_report['OnTime_Rate'] = (region_report['OnTime_Rate'] * 100).round(2)
        region_report['SLA_Met_Rate'] = (region_report['SLA_Met_Rate'] * 100).round(2)
        region_report = region_report.sort_values('Total_Revenue', ascending=False)

        st.dataframe(region_report, use_container_width=True)

        if st.button("📥 Download Regional Report"):
            st.markdown(create_download_link(
                region_report.reset_index(),
                "regional_performance_report.csv",
                "Regional Report"
            ), unsafe_allow_html=True)

    elif report_type == "Market Analysis":
        st.subheader("🌎 Market Analysis Report")

        market_report = filtered_df.groupby('Market').agg({
            'Sales': ['count', 'sum'],
            'Order Profit Per Order': 'sum',
            'delay_gap': 'mean',
            'On_Time': 'mean'
        }).round(2)

        market_report.columns = ['Total_Orders', 'Total_Revenue', 'Total_Profit', 'Avg_Delay', 'OnTime_Rate']
        market_report['OnTime_Rate'] = (market_report['OnTime_Rate'] * 100).round(2)
        market_report['Profit_Margin_%'] = (market_report['Total_Profit'] / market_report['Total_Revenue'] * 100).round(2)
        market_report = market_report.sort_values('Total_Revenue', ascending=False)

        st.dataframe(market_report, use_container_width=True)

        if st.button("📥 Download Market Report"):
            st.markdown(create_download_link(
                market_report.reset_index(),
                "market_analysis_report.csv",
                "Market Report"
            ), unsafe_allow_html=True)

    elif report_type == "Customer Segment Analysis":
        st.subheader("👥 Customer Segment Analysis")

        segment_report = filtered_df.groupby('Customer Segment').agg({
            'Sales': ['count', 'sum', 'mean'],
            'Order Profit Per Order': ['sum', 'mean'],
            'delay_gap': 'mean',
            'On_Time': 'mean',
            'Late_delivery_risk': 'sum'
        }).round(2)

        segment_report.columns = ['Total_Orders', 'Total_Revenue', 'Avg_Order_Value',
                                   'Total_Profit', 'Avg_Profit', 'Avg_Delay',
                                   'OnTime_Rate', 'High_Risk_Orders']
        segment_report['OnTime_Rate'] = (segment_report['OnTime_Rate'] * 100).round(2)
        segment_report = segment_report.sort_values('Total_Revenue', ascending=False)

        st.dataframe(segment_report, use_container_width=True)

        if st.button("📥 Download Segment Report"):
            st.markdown(create_download_link(
                segment_report.reset_index(),
                "customer_segment_report.csv",
                "Segment Report"
            ), unsafe_allow_html=True)

    elif report_type == "SLA Breach Details":
        st.subheader("⚠️ SLA Breach Analysis")

        sla_report = filtered_df.groupby(['SLA_Status', 'Shipping Mode']).agg({
            'Sales': ['count', 'sum'],
            'delay_gap': 'mean'
        }).round(2)

        sla_report.columns = ['Order_Count', 'Total_Revenue', 'Avg_Delay']
        sla_report = sla_report.reset_index()

        st.dataframe(sla_report, use_container_width=True)

        # SLA breach visualization
        fig_sla_detail = px.sunburst(
            sla_report,
            path=['SLA_Status', 'Shipping Mode'],
            values='Order_Count',
            color='Avg_Delay',
            color_continuous_scale='RdYlGn_r',
            title="SLA Status Breakdown by Shipping Mode"
        )
        fig_sla_detail.update_layout(height=600)
        st.plotly_chart(fig_sla_detail, use_container_width=True)

        if st.button("📥 Download SLA Report"):
            st.markdown(create_download_link(
                sla_report,
                "sla_breach_report.csv",
                "SLA Report"
            ), unsafe_allow_html=True)

    elif report_type == "Top Performers":
        st.subheader("🏆 Top Performers")

        col1, col2 = st.columns(2)

        with col1:
            st.markdown("**Top 10 by Revenue**")
            top_revenue = filtered_df.nlargest(10, 'Sales')[
                ['Shipping Mode', 'Order Region', 'Sales', 'Order Profit Per Order']
            ]
            st.dataframe(top_revenue, use_container_width=True)

        with col2:
            st.markdown("**Top 10 by Profit**")
            top_profit = filtered_df.nlargest(10, 'Order Profit Per Order')[
                ['Shipping Mode', 'Order Region', 'Sales', 'Order Profit Per Order']
            ]
            st.dataframe(top_profit, use_container_width=True)

        st.markdown("---")

        st.markdown("**Worst Performing Orders (Highest Delays)**")
        worst_delays = filtered_df.nlargest(10, 'delay_gap')[
            ['Shipping Mode', 'Order Region', 'delay_gap', 'SLA_Status', 'Sales']
        ]
        st.dataframe(worst_delays, use_container_width=True)


# ═══════════════════════════════════════════════════════════════════════════════
# FOOTER
# ═══════════════════════════════════════════════════════════════════════════════
st.markdown("---")
st.markdown("""
<div style="text-align: center; padding: 2rem 0; color: #7f8c8d;">
    <p><strong>APL Logistics Dashboard</strong> | Powered by Advanced Analytics</p>
    <p>📊 Real-time insights • 💰 Financial optimization • 🚚 Performance monitoring • 🔮 Predictive analytics</p>
    <p style="font-size: 0.9rem; margin-top: 1rem;">Built with decorators for performance optimization and error handling</p>
</div>
""", unsafe_allow_html=True)

Writing apl_enhanced_dashboard.py


## Step 6: Launch Dashboard

In [ ]:
from pyngrok import ngrok
import os

# Kill any existing streamlit processes
!pkill streamlit

# Start ngrok tunnel
public_url = ngrok.connect(8501)

print('=' * 70)
print('🚀 YOUR ENHANCED DASHBOARD IS LIVE!')
print('=' * 70)
print(f'\n📊 Dashboard URL: {public_url}')
print('\n' + '=' * 70)
print('⚠️  IMPORTANT: Keep this cell running!')
print('⚠️  Your dashboard will stop if you interrupt this cell')
print('=' * 70)
print('\n✨ NEW FEATURES:')
print('   • 🎨 Decorator functions for performance optimization')
print('   • 📈 Time series analysis with monthly trends')
print('   • 🔮 Predictive insights and risk assessment')
print('   • 💾 Export functionality for all reports')
print('   • 🎯 Enhanced filtering with date range')
print('   • 📊 Interactive correlation analysis')
print('   • 🏆 Top performers and worst delays tracking')
print('   • 📱 Modern UI with gradient styling')
print('   • 🔍 Advanced drill-down capabilities')
print('   • 📉 Distribution analysis with percentiles')
print('\n' + '=' * 70)

# Run Streamlit (this will keep running)
!streamlit run apl_enhanced_dashboard.py

🚀 YOUR ENHANCED DASHBOARD IS LIVE!

📊 Dashboard URL: NgrokTunnel: "https://spherulate-flo-procoercion.ngrok-free.dev" -> "http://localhost:8501"

⚠️  IMPORTANT: Keep this cell running!
⚠️  Your dashboard will stop if you interrupt this cell

✨ NEW FEATURES:
   • 🎨 Decorator functions for performance optimization
   • 📈 Time series analysis with monthly trends
   • 🔮 Predictive insights and risk assessment
   • 💾 Export functionality for all reports
   • 🎯 Enhanced filtering with date range
   • 📊 Interactive correlation analysis
   • 🏆 Top performers and worst delays tracking
   • 📱 Modern UI with gradient styling
   • 🔍 Advanced drill-down capabilities
   • 📉 Distribution analysis with percentiles




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.12.103.47:8501

2026-04-20 13:24:56.601 Please replace `use_container_width` with `width`.

`use_container_width` will be removed af